# AWQ on Qwen3-1.7B: detailed walkthrough

This notebook explains what the AWQ smoke test is doing, why each step exists, and how to scale it from a tiny sanity check to a larger Qwen3-1.7B experiment on Great Lakes.

The goal here is **not** to build a production INT4 inference engine. The goal is to reproduce the core AWQ calibration idea in plain PyTorch so we can inspect it, debug it, and connect it to the weight-outlier negative result and calibration/evaluation work.

Reference: Lin et al., **AWQ: Activation-aware Weight Quantization for On-Device LLM Compression and Acceleration**, arXiv:2306.00978.

## 0. Big-picture context

AWQ starts from one empirical observation:

> Some weight **input channels** are more important because the corresponding activation channels are large on real data.

A linear layer computes:

```text
Y = X W^T
```

where `X` is the layer input activation and `W` is the layer weight matrix with shape `(out_features, in_features)`.

AWQ uses calibration data to measure average activation magnitude per input channel:

```text
s_X[j] = mean(abs(X[:, j]))
```

Then it searches a simple scale family:

```text
s = s_X ** alpha,   alpha in [0, 1]
```

The AWQ objective for one linear layer is:

```text
min_s || Q(W diag(s)) diag(s)^-1 X - W X ||
```

In code, because the layer uses `X @ W.T`, this is implemented as:

```python
W_scaled = W * s[None, :]
W_q_scaled = pseudo_quantize(W_scaled)
W_awq = W_q_scaled / s[None, :]
```

The notebook uses `W_awq` as a dequantized analysis weight. It does **not** pack true INT4 kernels. That is deliberate: first we want to know whether the AWQ calibration/search path behaves sensibly on Qwen3-1.7B.

## 1. Notebook safety notes

- Start with a tiny target layer set. Full-model Qwen3-1.7B AWQ calibration can take time and memory.
- The default target is only the first transformer block's MLP projections.
- The notebook applies AWQ **temporarily** using a context manager. Original model weights are restored after the block exits.
- If you want persistent modified weights, do that later in a script after the smoke path is trusted.
- Run this notebook on a GPU compute node, not a login node.

In [ ]:
# Basic imports and repo path setup.
# This makes the notebook work whether Jupyter starts in the repo root or in notebooks/.

from pathlib import Path
import json
import pprint
import sys

REPO = Path.cwd()
if not (REPO / 'awq').exists():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Repo:', REPO)
print('Python:', sys.executable)

In [ ]:
# Import PyTorch, Hugging Face loaders, and local AWQ / metric utilities.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from awq import (
    collect_linear_inputs,
    pseudo_quantize_tensor,
    search_awq_scale,
    temporary_awq,
)
from weight_handles.metrics import compute_perplexity
from weight_handles.strata import load_strata, flatten_strata

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2. Configuration

On Great Lakes, Qwen3-1.7B should be available at the local scratch path below. This avoids repeatedly downloading from Hugging Face.

The default settings intentionally keep the run small:

- `MAX_LENGTH = 128`: short sequences for a fast smoke test.
- `MAX_TOKENS_PER_LAYER = 256`: store at most 256 activation rows per target layer.
- `N_GRID = 10`: test alpha values `0.0, 0.1, ..., 1.0`.
- `N_BITS = 4`: AWQ's common W4 setting.
- `GROUP_SIZE = 128`: common grouped weight-only setting from the paper.

After this works, increase these gradually.

In [ ]:
MODEL_NAME = '/scratch/huterer_root/huterer0/jiamingp/models/qwen3-1b7'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

MAX_LENGTH = 128
BATCH_SIZE = 1
MAX_TOKENS_PER_LAYER = 256

N_BITS = 4
GROUP_SIZE = 128
ZERO_POINT = True
N_GRID = 10

print('model:', MODEL_NAME)
print('device:', DEVICE)
print('dtype:', DTYPE)

## 3. Load model and tokenizer

This is the expensive first step. For Qwen3-1.7B on Great Lakes, it may take a few minutes because the model files are loaded from scratch into GPU memory.

We use `device_map='auto'` when CUDA is available so Hugging Face places the model on GPU. For this model, one A40 GPU should be enough for the notebook smoke test.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
model.eval()

first_param = next(model.parameters())
print('first parameter device:', first_param.device)
print('first parameter dtype:', first_param.dtype)

## 4. Calibration and evaluation texts

AWQ needs calibration text only to estimate activation magnitudes. It does not do backpropagation.

For a smoke test, the repo's small deterministic prompt strata are enough. For a real result, use more calibration data and held-out evaluation text.

Important distinction:

- **Calibration texts**: used to collect activations and search AWQ scales.
- **Evaluation texts**: used to measure whether the modified weights preserve behavior.

Do not over-interpret PPL from two short texts. Here it is a sanity check, not a benchmark.

In [ ]:
strata = load_strata()
all_texts = flatten_strata(strata)

# Use a small deterministic subset for the smoke test.
calib_texts = all_texts[:8]
eval_texts = all_texts[:4]

print('strata:', {k: len(v) for k, v in strata.items()})
print('n calibration texts:', len(calib_texts))
print('n eval texts:', len(eval_texts))
print('\nExample calibration text:\n', calib_texts[0])

## 5. Choose target layers

For one transformer block, Qwen-style MLP modules usually contain:

- `gate_proj`
- `up_proj`
- `down_proj`

This notebook starts with block 0 only. That keeps the activation buffers small and makes it easier to understand the outputs.

Later, set `RUN_ALL_MLP_LAYERS = True` to target every MLP projection.

In [ ]:
RUN_ALL_MLP_LAYERS = False

if RUN_ALL_MLP_LAYERS:
    target_layers = [
        name for name, module in model.named_modules()
        if any(name.endswith(suffix) for suffix in ['mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj'])
    ]
else:
    target_layers = [
        'model.layers.0.mlp.gate_proj',
        'model.layers.0.mlp.up_proj',
        'model.layers.0.mlp.down_proj',
    ]

print('n target layers:', len(target_layers))
for name in target_layers[:10]:
    module = model.get_submodule(name)
    print(name, tuple(module.weight.shape))

## 6. Collect layer input activations

`collect_linear_inputs` registers a forward pre-hook on each target `nn.Linear` module.

During a normal model forward pass, each hook sees the module input tensor. For a linear layer this has shape roughly:

```text
(batch, sequence_length, in_features)
```

The utility flattens batch and token dimensions into:

```text
(n_tokens_seen, in_features)
```

This flattened matrix is the `X` in the AWQ objective. Hooks are removed in a `finally` block inside the utility, so they should not accumulate across repeated runs.

In [ ]:
buffers = collect_linear_inputs(
    model,
    tokenizer,
    calib_texts,
    layer_names=target_layers,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    max_tokens_per_layer=MAX_TOKENS_PER_LAYER,
    device=DEVICE,
)

for name, X in buffers.items():
    print(name, tuple(X.shape), 'mean_abs=', float(X.abs().mean()))

## 7. Baseline behavior

Before modifying any weights, measure a baseline PPL on the tiny evaluation set.

Again: this PPL is only for checking that the pipeline runs. It is too small to be a real quality number.

In [ ]:
baseline_ppl = compute_perplexity(
    model,
    tokenizer,
    eval_texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    device=DEVICE,
)
baseline_ppl

## 8. Inspect AWQ search on one layer

Before applying AWQ to all target layers, inspect one layer directly.

For each alpha in the grid, the code:

1. computes `s = mean(abs(X)) ** alpha`, normalized for numerical stability;
2. scales input-channel columns of the weight: `W_scaled = W * s[None, :]`;
3. quantizes `W_scaled` groupwise;
4. folds the scale back into the weight: `W_awq = W_q_scaled / s[None, :]`;
5. measures output MSE between `X @ W.T` and `X @ W_awq.T`.

The selected alpha is the one with lowest output MSE on the cached activations.

In [ ]:
one_layer = target_layers[0]
one_module = model.get_submodule(one_layer)
one_X = buffers[one_layer]

one_result = search_awq_scale(
    one_module.weight.detach(),
    one_X,
    n_bits=N_BITS,
    group_size=GROUP_SIZE,
    zero_point=ZERO_POINT,
    n_grid=N_GRID,
)

print('layer:', one_layer)
print('best alpha:', one_result['alpha'])
print('best output MSE:', one_result['loss'])
print('scale shape:', tuple(one_result['scales'].shape))
print('AWQ dequantized weight shape:', tuple(one_result['weight'].shape))

## 9. Apply temporary AWQ to the selected layer set

`temporary_awq` applies AWQ dequantized weights inside a Python context manager:

```python
with temporary_awq(model, buffers, ...):
    # model uses AWQ weights here

# original weights restored here
```

This is safer than permanently overwriting the model while we are still testing.

In [ ]:
# Save one weight tensor before the context so we can verify restoration.
restore_check_layer = target_layers[0]
restore_check_before = model.get_submodule(restore_check_layer).weight.detach().float().cpu().clone()

with temporary_awq(
    model,
    buffers,
    layer_names=target_layers,
    n_bits=N_BITS,
    group_size=GROUP_SIZE,
    zero_point=ZERO_POINT,
    n_grid=N_GRID,
) as awq_results:
    awq_ppl = compute_perplexity(
        model,
        tokenizer,
        eval_texts,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

restore_check_after = model.get_submodule(restore_check_layer).weight.detach().float().cpu()
restored = torch.equal(restore_check_before, restore_check_after)

print('baseline_ppl:', baseline_ppl)
print('awq_ppl:', awq_ppl)
print('delta_ppl:', awq_ppl - baseline_ppl)
print('weights restored:', restored)
print('\nAWQ per-layer search results:')
pprint.pp(awq_results)

## 10. How to interpret the output

You should look at four things:

1. **Activation buffer shapes**
   - If a target layer is missing from `buffers`, the layer name is wrong or the layer did not execute.
   - Shape should be `(tokens_collected, in_features)`.

2. **Best alpha**
   - `alpha = 0.0` means the search preferred no activation-aware scaling for that layer.
   - Larger alpha means activation magnitude was useful for protecting important input channels.

3. **AWQ output MSE**
   - This is the layer-local objective, not global model quality.
   - Lower is better.

4. **PPL delta**
   - Since this smoke test modifies only the first block MLP, PPL may barely move.
   - If PPL explodes, something is wrong with the scale search, layer names, dtype/device handling, or evaluation path.

A confusing but normal outcome: AWQ can improve layer-local MSE but not obviously improve tiny-set PPL. The smoke test is mostly checking correctness and stability.

## 11. Scaling plan

Once the first-block test works, scale in this order:

1. Increase calibration tokens:
   ```python
   MAX_TOKENS_PER_LAYER = 512
   MAX_TOKENS_PER_LAYER = 1024
   ```

2. Add more layers:
   ```python
   RUN_ALL_MLP_LAYERS = True
   ```

3. Add attention projections after MLP works:
   ```python
   q_proj, k_proj, v_proj, o_proj
   ```

4. Move from notebook to script + sbatch for real runs.

The notebook is for understanding and debugging. Full experiments should be scripts so they can run unattended and save JSON results.

In [ ]:
# Optional: inspect all candidate MLP layer names without running AWQ on them.
# This is useful before setting RUN_ALL_MLP_LAYERS = True.

all_mlp_layers = [
    name for name, module in model.named_modules()
    if any(name.endswith(suffix) for suffix in ['mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj'])
]

print('n all MLP linear layers:', len(all_mlp_layers))
print('first 12:')
for name in all_mlp_layers[:12]:
    print(' ', name, tuple(model.get_submodule(name).weight.shape))

## 12. Common Great Lakes issues

**Jupyter job exits immediately**

Check:

```bash
sacct -j JOBID --format=JobID,JobName,State,ExitCode,Elapsed,MaxRSS
tail -n 120 logs/jupyter_awq_JOBID.err
```

**Notebook cannot import `awq`**

Make sure `PYTHONPATH=/home/jiamingp/llm-quant-experiments` is set. The sbatch launcher does this.

**Wrong model path**

Use:

```python
MODEL_NAME = '/scratch/huterer_root/huterer0/jiamingp/models/qwen3-1b7'
```

**Out of memory**

Lower `MAX_TOKENS_PER_LAYER`, reduce `target_layers`, or reduce `MAX_LENGTH`.

**Results are confusing**

Remember that the notebook is layer-local and tiny-data by default. The meaningful progression is:

```text
one layer -> one block -> all MLP -> all linear layers -> larger held-out eval
```